# Notebook 05: Panoptic Segmentation

**Module:** 09 Instance Segmentation  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Define panoptic segmentation task
2. Understand Panoptic Quality (PQ) metric
3. Know stuff vs things labeling
4. Design unified GeoSpatial scene understanding

## Panoptic Segmentation (Kirillov et al., 2019)

**Unified task:** Every pixel gets a label — either semantic class (stuff) or instance ID (things).

**Output format:** $(semantic\_id, instance\_id)$ per pixel

- Stuff pixels: instance_id = 0
- Thing pixels: unique instance_id per object

**No overlap:** Each pixel belongs to exactly one segment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Panoptic label map (concept): encode as category_id * 1000 + instance_id
# stuff: instance=0; things: instance=1,2,3...
H, W = 64, 64
panoptic = torch.zeros(H, W, dtype=torch.long)
panoptic[:, :20] = 1 * 1000 + 0      # road (stuff, class 1)
panoptic[10:30, 40:60] = 2 * 1000 + 1  # building instance 1
panoptic[35:55, 45:65] = 2 * 1000 + 2  # building instance 2

plt.imshow(panoptic.numpy(), cmap='nipy_spectral'); plt.title('Panoptic map (encoded)'); plt.colorbar(); plt.show()

## Panoptic Quality (PQ)

$$PQ = \underbrace{\frac{TP}{TP + \frac{1}{2}FP + \frac{1}{2}FN}}_{\text{SQ (Segmentation Quality)}} \times \underbrace{\frac{\sum IoU_{matched}}{TP}}_{\text{RQ (Recognition Quality)}}$$

Reported per class and as **mPQ** (mean PQ).

**Matching:** Predicted segment matches GT if same class and IoU > 0.5.

In [ ]:
# PQ components (conceptual)
TP, FP, FN = 8, 2, 3
matched_ious = torch.tensor([0.7, 0.8, 0.75, 0.9, 0.85, 0.6, 0.72, 0.88])
SQ = TP / (TP + 0.5*FP + 0.5*FN)
RQ = matched_ious.mean().item()
PQ = SQ * RQ
print(f'SQ={SQ:.3f}, RQ={RQ:.3f}, PQ={PQ:.3f}')

## GeoSpatial Panoptic Land Cover

Cityscapes-style: road (stuff) + individual buildings/trees (things) from satellite — full aquaculture landscape understanding.

---

## Summary

Panoptic = semantic + instance with PQ metric; no pixel overlap.

**Next:** [06_Panoptic_FPN.ipynb](06_Panoptic_FPN.ipynb)